In [1]:
# Packages to Install for Scraping
!pip -q install requests beautifulsoup4 
import requests, json
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import hashlib
import os
import re

import scraping_helpers


#Ensure that path for PDFs exists
os.makedirs(scraping_helpers.folder_name, exist_ok=True)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:


# Get the notice landing
landing_response = requests.get(scraping_helpers.notice_landing)
landing_soup = BeautifulSoup(landing_response.text, 'html.parser')

# Find the last page of notices: 
last_page = landing_soup.find("a",title="Go to last page").get("href")
#extract the number
match=re.search(r"page=(\d+)",last_page)
page_num = int(match.group(1))
#print(page_num)

# Loop through the notice pages
for p in range(page_num):
    page_path = scraping_helpers.notice_landing+f"?page={p}"
    #print(page_path)
    # Get the page into Beautiful soup:
    page_response = requests.get(page_path)
    #Check for success (troubleshooting) 
    #print(page_response.status_code)
    #print(len(page_response.text))
    page_soup = BeautifulSoup(page_response.text,'html.parser')
    # Pull out the notice IDs
    notice_container = page_soup.find("div", class_="department-components").find_all('div',class_="n-li")
    for notice in notice_container:
       
        rel_link = notice.find("a").get("href")
        #print(rel_link)
        # Pull out the Notice ID string
        match = re.search(r"/public-notices/(\d+)",rel_link)
        notice_id = match.group(1)
        # RUN THE EXTRACTION
        scraping_helpers.extract_notice(notice_id, scraping_helpers.log_path)
        




In [3]:
%pip -q install pandas langchain langchain-core langchain-community langchain-chroma langchain-huggingface chromadb sentence-transformers transformers accelerate sentencepiece langchain-docling
import pandas as pd

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from docling.chunking import HybridChunker
from langchain_docling import DoclingLoader
from pathlib import Path
import shutil
import re
from langchain_docling.loader import ExportType
from langchain_text_splitters import RecursiveCharacterTextSplitter

Note: you may need to restart the kernel to use updated packages.


In [4]:





# Get the latest records
latest_records = scraping_helpers.load_latest_records(scraping_helpers.log_path)
folder_ids = scraping_helpers.get_ids_from_folders(scraping_helpers.folder_name, scraping_helpers.log_path)

problem_ids = []

for notice_id in folder_ids:
    record = latest_records.get(notice_id)
    
    if record is None: 
        problem_ids.append((notice_id, "no log entry at all"))
        continue
    missing = [k for k in scraping_helpers.REQUIRED_FIELDS if k not in record]
    if missing:
        problem_ids.append((notice_id, f"missing {missing}"))
        continue
    
    record_metadata = {
           "notice_id": record["notice_id"],
            "title": record["title"],
            "cancelled": record["cancelled"],
            "public_testimony": record["public_testimony"],
            "notice_url": record["notice_url"],
            "posted_at": record["posted_at"],
            "event_datetime": record["event_datetime"],
            "address_1": record["address_1"],
            "address_2": record["address_2"],
            "status": record["status"],
            "checked_at": record["checked_at"],
    }
    #print(record)
    notice_files = record["files"]
    # TO UPDATE THE CHROMADB FOR PDF DATA
    for file in notice_files:
        # Skip files that didnt download
        if file["download_success"] == False:
            continue
        #Check if stale chunks from that file
        stale_chunks = scraping_helpers.vectorstore.get(where={
            "$and": [
                {"notice_id": record["notice_id"]},
                {"file_label": file["file_label"]}
            ]
             })
        # Delete if present
        if stale_chunks["ids"]:
            scraping_helpers.vectorstore._collection.delete(ids=stale_chunks["ids"])
        # Load to Docling 
        file_path = os.path.join(scraping_helpers.folder_name,record["notice_id"],file["file_label"])
        try:
            loader = DoclingLoader(
                file_path=file_path,
                export_type=scraping_helpers.EXPORT_TYPE,
                chunker=HybridChunker(tokenizer=scraping_helpers.EMBEDDING_MODEL)
            )
            docs = loader.load()
        # Load the docs
            for doc in docs:
                doc.metadata.pop("dl_meta", None)
                doc.metadata.pop("source", None)
                doc.metadata.update(record_metadata)
                doc.metadata.update({
                    "file_label": file["file_label"],
                    "file_hash": file["file_hash"],
                    "source_type":"pdf",
                })
            # Give the chunks labels
            ids = [f"{record['notice_id']}::{file['file_label']}::{i}" for i in range(len(docs))]
            scraping_helpers.vectorstore.add_documents(docs, ids=ids)
        
        except Exception as e:
            print(f"Failed to add Notice {record["notice_id"]} PDF {file["file_label"]}: {e}")
    # Now check for updated page text
    page_text = record["page_text"]
    text_hash = scraping_helpers.hash_sha256(page_text.encode("utf-8"))
    if page_text.strip() and not scraping_helpers.already_embedded(scraping_helpers.vectorstore, record["notice_id"], text_hash=text_hash):
        stale_text = scraping_helpers.vectorstore.get(where={
            "$and": [
                {"notice_id":record["notice_id"]},
                {"source_type":"page_text"}
            ]
             
        })
        # If stale, remove
        if stale_text["ids"]:
            scraping_helpers.vectorstore._collection.delete(ids=stale_text["ids"])

        try:
            page_docs = text_splitter.create_documents(
                texts=[record["page_text"]],
                metadatas=[{
                    **record_metadata,
                    "text_hash":text_hash,
                    "source_type":"page_text",
                }],
            )
            ids = [f"{record['notice_id']}::pagetext::{text_hash}::{i}" for i in range(len(page_docs))]
            scraping_helpers.vectorstore.add_documents(page_docs, ids=ids)
        except Exception as e:
            print(f"Failed to add Notice {record["notice_id"]} page text: {e}")
        # When done, print that the notice has been added/ updated can comment out when done troubleshooting
        #print(f"Notice {notice_id} has been added to Chromadb\n")
    
    

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-08-03 14:01:01,797 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:01:01,808 [RapidOCR] download_file.py:60: File exists and is valid: /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 14:01:01,808 [RapidOCR] main.py:63: Using /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 14:01:01,844 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:01:01,846 [RapidOCR] download_file.py:60: File exists and is valid: /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 14:01:01,846 [RapidOCR] main.py:63: Using /opt/anaconda3/lib/python3.13/sit

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Failed to add Notice 16492916 page text: name 'text_splitter' is not defined


[INFO] 2026-08-03 14:01:04,911 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:01:04,919 [RapidOCR] download_file.py:60: File exists and is valid: /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 14:01:04,919 [RapidOCR] main.py:63: Using /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 14:01:04,941 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:01:04,942 [RapidOCR] download_file.py:60: File exists and is valid: /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 14:01:04,942 [RapidOCR] main.py:63: Using /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 14:01:04,964 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:01:04,980 [RapidOCR] download_file.py:60: File exists and is val

...

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Failed to add Notice 16602151 page text: name 'text_splitter' is not defined


[INFO] 2026-08-03 14:06:30,377 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:06:30,389 [RapidOCR] download_file.py:60: File exists and is valid: /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 14:06:30,390 [RapidOCR] main.py:63: Using /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 14:06:30,421 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:06:30,424 [RapidOCR] download_file.py:60: File exists and is valid: /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 14:06:30,424 [RapidOCR] main.py:63: Using /opt/anaconda3/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 14:06:30,448 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 14:06:30,466 [RapidOCR] download_file.py:60: File exists and is val

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Failed to add Notice 16602301 page text: name 'text_splitter' is not defined


In [5]:
# Check how many records added 
print(f"Total Records: {scraping_helpers.vectorstore._collection.count()}")

Total Records: 399


In [6]:
print(f"{len(problem_ids)} problem notice(s) out of {len(folder_ids)} folders")
for nid, reason in problem_ids:
    print(nid, "-", reason)

0 problem notice(s) out of 70 folders
